# Ft. Huachucah Methods
This is a review of the methods used to calculate biomass in the Ft. Huachucah area.

## Field Collection Methods



In [ ]:
# import calc_biomass package
import sys
sys.path.append('../')
from calc_biomass import manage_data

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Jupyter magic to make plots display interactive
# must install ipympl (Ipython-matplotlib) and nodejs
from ipywidgets.embed import embed_minimal_html
%matplotlib widget

## Canopy Biomass Calculation Methods
### Calculating DRC from DBH
If a tree had a central stem, regardless of species, diameter was measured at DBH. But if the tree had multiple stems from a common root base, DRC (diameter at root collar was measured). Which measurement was used was recorded in the dataset. Most allometric equations require either DBH or DRC, so to be consistent within a species, the following equation was applied to the Excel spreadsheet to convert DBH to DRC for oak, pinyon, and juniper species.

    Chojnacky, D. 1988. Juniper, Pinyon, Oak, and Mesquite Volume Equations for Arizona. USDA Forest Service, Intermountain Research Station, Research Paper INT-RP-391. Ogden, UT.

### Calculating Equivalent DRC

Using standard protocol for Diameter at Root Collar (DRC), when multiple stems joined to a single root collar below the soil, individual stems were measured for the same tree. These multiple basal measurements must be combined into a single measurement using the equation:

equivalent diameter = $\sqrt{\sum_{i=1}^{n}}RCD_{i}^{2}$

This equation is provided in both of:
    
    Chojnakcy, D.C. 1992 . Estimating volume and biomass for dryland oak speices. In: Ffolliott P.F., Gottfried, G.J., Bennett, D.A., Hernandez, C.V.-M., Ortega-Rubio, A., and R.H. Hamre, technical coordinators. Ecology and management of oak and associated woodlands: perspectives in the southwestern United States and Northern Mexico. Proceedings April 7-30, 1992; Sierra Vista, AZ. USDA Forest Service, Rocky Mountain Forest and Range Experiment Station General Technical Report RM-218:155-161.
    
    Grier, C.C., Elliott, K.J., McCullough, D.G. 1992. Biomass distribution and productivity of Pinus edulis-Juniperus monosperma woodlands of north-central Arizona. Forest Ecology and Management, 50:331-350.

In [ ]:
def calc_edrc(df):
    # df is one tree id from a single plot
    # for trees with only one tree id, the sqrt(d2) == d
    dbh = (sum(df['dbh']**2))**0.5
    
    other_val = df.max()

    newdf = pd.DataFrame()
    newdf.loc[1,'DBH'] = dbh
    newdf.loc[1, df.columns] = other_val

    return newdf

In [ ]:
# load data
data = manage_data.LoadData()
data.load_data(data.data_path)

# calculate ERC
data.df.rename(columns={'Final DBH/DRC':'dbh', 'spp':'Species','Species Code':'spp'}, inplace=True)
data.df = data.df.groupby(['Plot', 'Tree Number']).apply(calc_edrc).reset_index()

In [27]:
# Error noted below, now the new df with it's new index needs to have units reassigned, but the column names are all changed around
data.units['input']['DBH'] = data.units['input'].pop('Final DBH/DRC')
data.check_df_units('input')

### Cleaning the data
A number of quality assurance procedures applied in the other Ft. Huachuca Jupyter notebooks identified the following data cleaning requirements.

In [ ]:
# clean data

# missing species for generic pinyon
pied = data.df.spp.isna() & (data.df.Species=='Pinyon')
data.df.loc[pied, 'spp'] = 'PIED'

# height for mesquite are in cm, not clinometer %
prve = (data.df['spp']=='PRVE')&(data.df['Height']<=0)
data.df.loc[prve, 'Height'] = data.df.loc[prve, 'Base reading']/100

# accidentally flipped the sign of a juniper height clinometer %
jude = (data.df['spp']=='JUDE2')&(data.df['Height']<0)
ht = (-data.df.loc[jude, 'Top Reading']- data.df.loc[jude, 'Base reading']) / 100 * data.df.loc[jude, 'Distance (M)']
data.df.loc[jude, 'Height'] = ht

# quarter inch drc tree (actual circumference 200 cm)
qrt_in = (data.df['spp']=='QUEM')&(data.df['DBH']<1)
c = 200
d = c/np.pi
d_in = d/2.54
data.df.loc[qrt_in, 'DBH'] = d_in
data.df.loc[qrt_in, 'DBH/DRC Circumfrence (cm)'] = 200

# 3m DRC (circumference 94, not 944)
m3 = (data.df['spp']=='QUOB')&(data.df['DBH']<117)
dbh_cm = 94/np.pi
dbh_in = dbh_cm / 2.54
data.df.loc[m3, 'DBH'] = dbh_in
data.df.loc[m3, 'DBH/DRC Circumfrence (cm)'] = 94

### Calculating biomass

#### Allometric equations

##### Oaks
    Chojnakcy, D.C. 1992 . Estimating volume and biomass for dryland oak speices. In: Ffolliott P.F., Gottfried, G.J., Bennett, D.A., Hernandez, C.V.-M., Ortega-Rubio, A., and R.H. Hamre, technical coordinators. Ecology and management of oak and associated woodlands: perspectives in the southwestern United States and Northern Mexico. Proceedings April 7-30, 1992; Sierra Vista, AZ. USDA Forest Service, Rocky Mountain Forest and Range Experiment Station General Technical Report RM-218:155-161.

##### Pinyon and Juniper

    Grier, C.C., Elliott, K.J., McCullough, D.G. 1992. Biomass distribution and productivity of Pinus edulis-Juniperus monosperma woodlands of north-central Arizona. Forest Ecology and Management, 50:331-350.

1 hour fuels use branchwood adjustment factor from FuelCalc:

    Lutes, D. 2020. FuelCalc User's Guide (version 1.7). USDA Forest Service, Rocky Mountain Research Station.


##### Pines

Equations used require tree volume from the BC Ministry of Forestry volume tables.

    Standish, J.T., Manning, G.H., and Demaerschalk, J.P. 1985. Development of biomass equations for British Columbia tree species. Canadian Forestry Service, Pacific Forest Research Centre, Information Report BC-X-264 (Vancouver, BC)

    Susan Watts eds. 1983. Forestry Handbook for British Columbia, 4th Edition. University of British Columbia. Forest Club.
    

In [ ]:
wt = manage_data.CalcBiomass(data)
# what equations should be used for these species?
wt.add_equ_column()

In [7]:
wt.calc_biomass('total')

150: RuntimeWarning: invalid value encountered in log10
150: RuntimeWarning: invalid value encountered in log10


In [8]:
wt.calc_biomass('foliage')

150: RuntimeWarning: invalid value encountered in log10


In [9]:
wt.calc_biomass('avl_canfuel')

150: RuntimeWarning: invalid value encountered in log10
150: RuntimeWarning: invalid value encountered in log10
